# NB00: Installation & Environment Verification

**Purpose**: Verify that your RELIC environment is correctly set up before running any tutorials.

This notebook is analogous to WRF's "compile verification" step: it checks that the
runtime dependencies declared in `pyproject.toml` are installed at the declared minimum
versions, that RELIC's package structure is importable, and that the core
simulation chain (Functor -> Simulator -> State -> Output) works end-to-end.

**Prerequisites**:
- Windows with PowerShell
- Anaconda / Miniconda installed
- `RELIC` conda environment created

**Time**: ~2 minutes


## Architecture Anchor

This notebook follows `docs\source\architecture_overview.md`.
Steps 1-4 check the Python environment, the runtime dependencies declared in `pyproject.toml`, the RELIC package imports and the RELIC device policy (`relic.core.device_policy`).
Step 5 defines a minimal notebook Functor on the `BiologicalProcessFunctor` bridge and runs it through the actual `RELICSimulator` and `StateManager`.
That Functor computes a scalar test value; it is not a scientific process.

## Step 1: System Information

Check Python version, OS, and conda environment.


In [1]:
import asyncio
import os
import platform
import sys
from pathlib import Path

if hasattr(asyncio, "WindowsSelectorEventLoopPolicy"):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# --- Environment Guard (prevent OpenMP duplicate lib crash) ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

PROJECT_ROOT = Path(os.environ.get("RELIC_PROJECT_ROOT", Path.cwd())).resolve()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=== System Information ===")
print(f"  Python:    {sys.version}")
print(f"  Interpreter: {sys.executable}")
print(f"  Platform:  {platform.platform()}")
print(f"  Arch:      {platform.machine()}")
print(f"  Conda env: {os.environ.get('CONDA_DEFAULT_ENV', 'NOT SET')}")
print(f"  CWD:       {os.getcwd()}")
print(f"  Project:   {PROJECT_ROOT}")
print()

env_name = os.environ.get("CONDA_DEFAULT_ENV", "")
if "RELIC" in env_name.upper():
    print("[OK] Running inside RELIC conda environment")
else:
    print(f"[WARNING] Expected RELIC env, got: '{env_name}'")
    print("  Run: conda activate RELIC")

=== System Information ===
  Python:    3.11.13 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:03:15) [MSC v.1929 64 bit (AMD64)]
  Interpreter: python
  Platform:  Windows-10-10.0.26200-SP0
  Arch:      AMD64
  Conda env: <CONDA_ENV_RELIC>
  CWD:       tutorial\primary
  Project:   <RELIC_ROOT>

[OK] Running inside RELIC conda environment


## Step 2: Dependencies

The cell reads the runtime dependencies declared in `pyproject.toml` (`[project].dependencies`) and checks that the installed version of each distribution satisfies its declared specifier (using the stored specifier for that distribution).
`importlib.metadata.version()` raises `PackageNotFoundError` for a distribution that is not installed, and the cell raises `RuntimeError` that lists every installed version outside its specifier.
For `dask[complete]` only the `dask` distribution is checked, not the packages that the `complete` extra installs.
The cell then imports eight core packages to confirm that they load.


In [2]:
import importlib
import tomllib
from importlib import metadata

from packaging.requirements import Requirement  # installed with ipykernel

# Runtime dependencies declared in pyproject.toml ([project].dependencies).
with (PROJECT_ROOT / "pyproject.toml").open("rb") as handle:
    declared = [Requirement(text) for text in tomllib.load(handle)["project"]["dependencies"]]

print(f"=== Declared Runtime Dependencies (pyproject.toml, {len(declared)} entries) ===")
unsatisfied = []
for requirement in declared:
    if requirement.marker is not None and not requirement.marker.evaluate():
        print(f"  [SKIP] {requirement.name:<14s} environment marker not met: {requirement.marker}")
        continue
    # metadata.version() raises PackageNotFoundError for a distribution that is not installed.
    installed = metadata.version(requirement.name)
    if requirement.specifier.contains(installed, prereleases=True):
        print(f"  [OK] {requirement.name:<14s} {str(requirement.specifier):<10s} installed {installed}")
    else:
        print(f"  [ERROR] {requirement.name:<14s} {str(requirement.specifier):<10s} installed {installed}")
        unsatisfied.append(f"{requirement.name} {installed} (requires {requirement.specifier})")
if unsatisfied:
    raise RuntimeError("Installed versions outside pyproject.toml specifiers: " + "; ".join(unsatisfied))

# Import check for eight core packages (module import names).
CORE_PACKAGES = [
    ("torch", "PyTorch (differentiable framework)"),
    ("numpy", "NumPy (array operations)"),
    ("scipy", "SciPy (scientific computing)"),
    ("xarray", "xarray (N-D labeled arrays)"),
    ("yaml", "PyYAML (configuration)"),
    ("pydantic", "Pydantic (data validation)"),
    ("matplotlib", "Matplotlib (visualization)"),
    ("netCDF4", "netCDF4 (I/O)"),
]

print()
print("=== Core Package Imports ===")
for module_name, description in CORE_PACKAGES:
    importlib.import_module(module_name)
    print(f"  [OK] {description:<40s} import {module_name}")

print()
print(f"[OK] {len(declared)} declared runtime dependencies satisfy pyproject.toml; "
      f"{len(CORE_PACKAGES)} core packages imported")

=== Declared Runtime Dependencies (pyproject.toml, 25 entries) ===
  [OK] torch          >=2.0      installed 2.13.0+cu130
  [OK] numpy          >=1.24     installed 2.3.2
  [OK] scipy          >=1.10     installed 1.16.1
  [OK] statsmodels    >=0.14     installed 0.14.6
  [OK] xarray         >=2023.1   installed 2025.7.1
  [OK] zarr           >=3.0      installed 3.1.3
  [OK] h5py           >=3.10     installed 3.14.0
  [OK] dask           >=2023.1   installed 2026.8.0
  [OK] hydra-core     >=1.3      installed 1.3.4
  [OK] pydantic       >=2.0      installed 2.11.7
  [OK] jsonschema     >=4.18     installed 4.25.0
  [OK] pyyaml         >=6.0      installed 6.0.2
  [OK] toml           >=0.10     installed 0.10.2
  [OK] cf-units       >=3.2      installed 3.3.1
  [OK] netcdf4        >=1.6      installed 1.7.4
  [OK] plotly         >=5.0      installed 6.2.0
  [OK] dash           >=2.0      installed 3.2.0
  [OK] matplotlib     >=3.7      installed 3.10.0
  [OK] seaborn        >=0.12   

  [OK] PyTorch (differentiable framework)       import torch
  [OK] NumPy (array operations)                 import numpy
  [OK] SciPy (scientific computing)             import scipy


  [OK] xarray (N-D labeled arrays)              import xarray
  [OK] PyYAML (configuration)                   import yaml
  [OK] Pydantic (data validation)               import pydantic


  [OK] Matplotlib (visualization)               import matplotlib
  [OK] netCDF4 (I/O)                            import netCDF4

[OK] 25 declared runtime dependencies satisfy pyproject.toml; 8 core packages imported


## Step 3: RELIC Package Import Verification

Test that all top-level RELIC submodules can be imported.
This is the equivalent of checking that `wrf.exe` and `real.exe` were compiled successfully.


In [3]:
import importlib
import relic

print(f"=== RELIC Package v{relic.__version__} ===")
print(f"  Phase: {relic.__phase__}")
print()

# Imported by `import relic` itself (relic/__init__.py imports these eagerly).
CORE_IMPORTS = [
    ("relic.core", "Core framework"),
    ("relic.core.functor", "FunctorBase & bridge classes"),
    ("relic.core.simulator", "RELICSimulator"),
    ("relic.core.state", "StateManager"),
    ("relic.core.abstractions", "Four-pillar abstractions"),
]

print("--- Core Framework (Eager) ---")
for module_path, description in CORE_IMPORTS:
    importlib.import_module(module_path)
    print(f"  [OK] {description:<35s} ({module_path})")

# Every other top-level package under relic/.
SUBMODULES = [
    ("relic.scientific", "Scientific processes (7 domains)"),
    ("relic.coupling", "Coupling & multi-scale integration"),
    ("relic.data", "Data I/O & standards"),
    ("relic.parameters", "Parameter management"),
    ("relic.optimization", "Calibration & sensitivity"),
    ("relic.computation", "Solvers, operators, GPU/parallel"),
    ("relic.stochastic", "Stochastic & uncertainty"),
    ("relic.interfaces", "CLI, API, web interfaces"),
    ("relic.extensions", "Plugin extensions"),
    ("relic.monitoring", "Prometheus metrics"),
    ("relic.models", "Basin/patch/landscape, surrogates"),
    ("relic.contracts", "Variable contracts (SCM)"),
    ("relic.utils", "Physical constants & helpers"),
    ("relic.validation", "Testing & conservation checks"),
    ("relic.orchestration", "Graph IR & compilation"),
]

print()
print("--- Submodules ---")
for module_path, description in SUBMODULES:
    importlib.import_module(module_path)
    print(f"  [OK] {description:<35s} ({module_path})")

print(f"\n  Imported {len(SUBMODULES)}/{len(SUBMODULES)} submodules")

=== RELIC Package v1.0.0 ===
  Phase: Phase 1 - Core Framework Integration

--- Core Framework (Eager) ---
  [OK] Core framework                      (relic.core)
  [OK] FunctorBase & bridge classes        (relic.core.functor)
  [OK] RELICSimulator                      (relic.core.simulator)
  [OK] StateManager                        (relic.core.state)
  [OK] Four-pillar abstractions            (relic.core.abstractions)

--- Submodules ---


  [OK] Scientific processes (7 domains)    (relic.scientific)
  [OK] Coupling & multi-scale integration  (relic.coupling)
  [OK] Data I/O & standards                (relic.data)
  [OK] Parameter management                (relic.parameters)
  [OK] Calibration & sensitivity           (relic.optimization)
  [OK] Solvers, operators, GPU/parallel    (relic.computation)
  [OK] Stochastic & uncertainty            (relic.stochastic)
  [OK] CLI, API, web interfaces            (relic.interfaces)
  [OK] Plugin extensions                   (relic.extensions)
  [OK] Prometheus metrics                  (relic.monitoring)
  [OK] Basin/patch/landscape, surrogates   (relic.models)
  [OK] Variable contracts (SCM)            (relic.contracts)
  [OK] Physical constants & helpers        (relic.utils)
  [OK] Testing & conservation checks       (relic.validation)
  [OK] Graph IR & compilation              (relic.orchestration)

  Imported 15/15 submodules


## Step 4: PyTorch & Device Detection

Check CUDA availability and the device that RELIC's device policy selects.
This determines whether your simulations can run on GPU.
`resolve_device("auto")` from `relic.core.device_policy` is the call the simulator makes for its default setting `device: "auto"`: CUDA when it is available, otherwise CPU.

The stderr line about triton comes from PyTorch (`torch.utils.flop_counter`) when `torch.compile` loads; triton is not installed in the RELIC environment and the `eager` backend used here does not need it.

In [4]:
import torch
from relic.core.device_policy import is_cuda_available, resolve_device

print("=== PyTorch & Device Detection ===")
print(f"  PyTorch version:  {torch.__version__}")
print(f"  CUDA available:   {is_cuda_available()}")
print(f"  CUDA version:     {torch.version.cuda if is_cuda_available() else 'N/A'}")

if is_cuda_available():
    print(f"  GPU count:        {torch.cuda.device_count()}")
    for gpu_index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(gpu_index)
        memory_gb = properties.total_memory / 1e9
        print(f"  GPU {gpu_index}: {properties.name} ({memory_gb:.1f} GB)")
else:
    print("  GPU:              Not available (will use CPU)")

# Same policy call as the simulator setting device: "auto" (CUDA first, CPU fallback).
default_device = resolve_device("auto")
print(f"  RELIC 'auto' device: {default_device}")

x = torch.tensor([1.0, 2.0, 3.0], device=default_device)
y = x * 2
print(f"\n  Tensor test: {x.tolist()} * 2 = {y.tolist()} on {default_device}")

torch_compile_available = hasattr(torch, "compile")
print(f"  torch.compile attribute present: {torch_compile_available}")
if torch_compile_available:
    compiled_fn = torch.compile(lambda tensor: tensor + 1, backend="eager")
    compile_result = compiled_fn(torch.tensor(1.0))
    print(f"  torch.compile smoke result: {float(compile_result):.1f}")

=== PyTorch & Device Detection ===
  PyTorch version:  2.13.0+cu130
  CUDA available:   True
  CUDA version:     13.0
  GPU count:        1
  GPU 0: NVIDIA GeForce RTX 5090 Laptop GPU (25.7 GB)
  RELIC 'auto' device: cuda



  Tensor test: [1.0, 2.0, 3.0] * 2 = [2.0, 4.0, 6.0] on cuda
  torch.compile attribute present: True


W0930 12:50:48.392000 84528 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


  torch.compile smoke result: 2.0


## Step 5: Smoke Test -- Minimal Functor Chain

The most important verification: can we create a Functor, register it with a Simulator,
run one timestep, and get an output? This is the RELIC equivalent of running `wrf.exe`
with an idealized test case.

- `SmokeTestFunctor.forward` has the Functor kernel signature `forward(state, time, dt, **kwargs)`.
  The simulator calls the Functor object; `FunctorBase.__call__()` validates the state, calls `forward()` with `time`, `dt` and any extra keyword context, and runs the local conservation and validation hooks.
- Every initial-state variable needs declared units (SCM contract, SSOT variable catalog, `validation.numerical.state_variable_bounds`, or the simulator setting `state_units`).
  The two test variables exist only in this notebook, so `state_units` declares them dimensionless (`"1"`). Without a declaration, `initialize_state()` raises `ValueError`.
- `forward()` returns a plain tensor for `verification_output`. This simulator has no `run_specs` (no `irf_specs_path` in its config) and the name does not end in `_flux` or `_rate`, so the simulator commits the tensor as an `ACCUMULATE` update: the returned value is added to the stored value.
  For a rate-like variable, or for any plain tensor when the simulator has `run_specs`, the default mode is `REPLACE`.
  `verification_output` starts at 0, so one step gives 0.1 x 25 = 2.5.

The cell prints no RELIC log lines. RELIC modules log through `logging.getLogger(__name__)` and leave the logging configuration to the application: importing `relic.scientific` in Step 3 fills the Functor registry (`register_all_functors()`) without configuring logging, and the registry does not import the command-line entry modules (`*_cli.py`), which configure logging only inside their `main()`. To see the INFO messages of registration and state initialization, configure logging first, for example with `logging.basicConfig(level=logging.INFO)`. Importing `relic.interfaces.cli.main` also leaves logging to the application; only its command-line `main()` configures the root handler.

In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from relic.core.functor import (
    BiologicalProcessFunctor,
    ExecutionPhase,
    FunctorMetadata,
    ProcessCategory,
)
from relic.core.functor.functor_types import SpatialScale, VariableKind
from relic.core.simulator import RELICSimulator


class SmokeTestFunctor(BiologicalProcessFunctor):
    """Minimal bridge-based functor for installation verification."""

    def __init__(self, name: str = "smoke_test") -> None:
        metadata = FunctorMetadata(
            name=name,
            description="Scalar verification functor for notebook environment checks.",
            category=ProcessCategory.BIOLOGICAL_PROCESS,
            execution_phase=ExecutionPhase.MAIN_PROCESSES,
            mathematical_expression=(
                "verification_output(t + dt) = verification_output(t) + 0.1 * verification_signal(t)"
            ),
            references=["None (scalar smoke test for installation verification)"],
            input_variables=["verification_signal"],
            output_variables=["verification_output"],
            spatial_scale=SpatialScale.PATCH,
            variable_kind=VariableKind.STATE,
        )
        super().__init__(name=name, config={}, metadata=metadata)

    # No get_dependencies() override: the bridge default returns
    # self._merge_declared_dependencies(), so instance-declared dependencies
    # (config["dependencies"], set_dependencies, add_dependency) are kept.

    def forward(self, state, time, dt, **kwargs):
        # References:
        #   - None (scalar smoke test for installation verification)
        # Mathematical expression:
        #   - verification_output(t + dt) = verification_output(t) + 0.1 * verification_signal(t)
        #     forward() returns the increment 0.1 * verification_signal as a plain tensor.
        #     The simulator adds a plain tensor to the stored value (ACCUMULATE) only
        #     when it has no run_specs (no irf_specs_path in its config) and the variable
        #     is not rate-like. A name ending in "_flux" or "_rate", or an output
        #     specification with is_rate=True, gives REPLACE; with run_specs every plain
        #     tensor starts as REPLACE and RunSpecs may set another mode (see
        #     relic.core.simulator.execution_group._coerce_runtime_update_item).
        #     This simulator has no run_specs and verification_output is not rate-like.
        # Parameter meanings:
        #   - verification_signal: scalar test input read from the state
        #   - 0.1: fixed factor of this smoke test (a notebook literal, not a parameter-database entry)
        # Units:
        #   - verification_signal and verification_output are dimensionless ("1"),
        #     declared through the simulator setting state_units below.
        # Domain/constraints:
        #   - verification_signal must be a finite floating torch.Tensor.
        # Sign convention:
        #   - a positive signal increases verification_output.
        signal = state["verification_signal"]
        return {"verification_output": signal * 0.1}


# The two test variables have no SCM contract or SSOT catalog entry, so the
# simulator setting state_units declares their units; initialize_state()
# raises ValueError for any initial-state variable without declared units.
sim_test = RELICSimulator(
    config={
        "device": "auto",
        "strict_dependencies": False,
        "state_units": {"verification_signal": "1", "verification_output": "1"},
    }
)
functor = SmokeTestFunctor()
sim_test.register_functor(functor, execution_group="fast")

sim_test.initialize_state(
    {
        "verification_signal": torch.tensor(25.0, dtype=torch.float32),
        "verification_output": torch.tensor(0.0, dtype=torch.float32),
    }
)

result = sim_test.step(dt=1800.0)
current_state = sim_test.state_manager.get_current_state()
output_val = current_state["verification_output"]
expected_val = torch.tensor(2.5, dtype=torch.float32, device=output_val.device)

print("=== Smoke Test Results ===")
print(f"  Input:  verification_signal = {float(current_state['verification_signal']):.1f}")
print(f"  Output: verification_output = {float(output_val):.2f}")
print(f"  Expected:                   = {float(expected_val):.2f}")
print(f"  State device:               {output_val.device}")
print()

if torch.allclose(output_val, expected_val, atol=1.0e-6):
    print("[OK] Smoke test PASSED -- RELIC is working correctly!")
else:
    raise AssertionError(
        f"Smoke test failed: expected {float(expected_val):.2f}, got {float(output_val):.2f}"
    )

summary = sim_test.get_functor_summary()
print(f"\n  Registered functors: {len(summary)}")
for name, info in summary.items():
    print(
        f"    {name}: group={info['execution_group']}, scale={info['spatial_scale']}, "
        f"kind={info['variable_kind']}, inputs={info['input_variables']}"
    )


=== Smoke Test Results ===
  Input:  verification_signal = 25.0
  Output: verification_output = 2.50
  Expected:                   = 2.50
  State device:               cuda:0

[OK] Smoke test PASSED -- RELIC is working correctly!

  Registered functors: 1
    smoke_test: group=fast, scale=patch, kind=state, inputs=['verification_signal']


## Step 6: RELIC Architecture Overview

Now that your environment is verified, here is a map of the RELIC system.
`relic.print_architecture_guide()` prints static text stored in `relic\__init__.py`; the text is not generated from the code.
These entries of the guide describe more than the code provides:
- Isotopes "Tracking": `relic.scientific.isotopes` has no isotope tracking runtime. `wire_isotope_tracker`, `IsotopeTrackerService` and `IsotopeHeartbeatFunctor` raise an error marked `[N]` (runtime unbound); only the 13C, 15N and 18O fractionation calculators in `relic.scientific.isotopes.fractionation` run.
- Data "Standards": `relic.data.standards` provides unit conversion and CF conventions (its package exports are `CFConventions` and `CFAttributeSynchronizer`) and no CMIP standard. CMIP6 appears only as the data adapter `CMIP6Adapter` and its variable-name mapping in `relic.coupling.external`.
- Data "Quality" and "Transformers": `relic.data.quality` contains validators only (no cleaning or gap filling) and `relic.data.transformers` contains no transformer; both are experimental placeholder packages that import only with `RELIC_EXPERIMENTAL_DATA_QUALITY=1` or `RELIC_EXPERIMENTAL_DATA_TRANSFORMERS=1`.
- "Validation: L1-L4": the validation framework defines the layers L1-L4, but validation rules are registered only for L1 and L2; `relic.validation.unit_tests`, `integration_tests`, `benchmarks` and `real_world_cases` contain no code.
- "Interfaces: ... notebooks": `relic.interfaces.notebooks` holds one comment line and no code. The interfaces with code are the CLI (`relic.interfaces.cli`), the REST API (`relic.interfaces.api`) and the web app (`relic.interfaces.web`).
- "Monitoring": `relic.monitoring` exports only Prometheus metric functions; it has no Grafana dashboards and no execution logging.

Other packages mark parts of their runtime as `[N]` in the same way, for example model execution in `relic.optimization`, and `EmissionInventory` and `AtmosphereCoupling` in `relic.scientific.anthroposphere.emissions`. For those packages the guide names the component families, not complete end-to-end runs.

In [6]:
import relic

relic.print_architecture_guide()

# Learning path of tutorial\primary (topics and main APIs of each notebook).
LEARNING_PATH = [
    ("Beginner (NB01-05)", [
        ("01", "Minimal run", "Functor -> Simulator -> State -> Output"),
        ("02", "Real components", "ParameterManager, ContractRegistry, real Functors"),
        ("03", "Coupling", "TimeStagger, CouplingCapabilityRegistry, HealthChecker"),
        ("04", "CLI", "create_main_parser, run_layered_simulation"),
        ("05", "Hydro + Pedo", "BucketSoilMoisture, FirstOrderDecomposition, CarbonPoolsFunctor"),
    ]),
    ("Intermediate (NB06-12)", [
        ("06", "Scale topology", "SpatialScale, TemporalScale, execution groups and plans"),
        ("07", "Extensions", "Extensions, monitoring, physical constants"),
        ("08", "Orchestration", "GraphIR, SSOTCatalog, CF metadata, I/O health"),
        ("09", "Optimization", "Optimizers, samplers, CRPS, device selection"),
        ("10", "Composition", "Five hydrosphere Functors, DependencyGraphResolver"),
        ("11", "Conservation", "DEC audit, transactional updates, clip_loss"),
        ("12", "Numerics", "Operator splitting, adaptive dt, event triggers"),
    ]),
    ("Advanced Theory (NB13-15)", [
        ("13", "Structuralism", "Epistemology, Lyapunov, structural invariants"),
        ("14", "DEC topology", "Incidence matrix, left null space, topology signature"),
        ("15", "Category theory", "Objects, morphisms, scale lattice, adjunctions"),
    ]),
    ("Practical Workflows (NB16-21)", [
        ("16", "Data preprocessing", "Forcing data, QC, CF metadata"),
        ("17", "Tropical scenario", "Generated forcing, production Functors end-to-end"),
        ("18", "Module gallery", "All 7 scientific domains"),
        ("19", "Post-processing", "Visualization, evaluation, diagnostics"),
        ("20", "Calibration", "Sensitivity, optimization, uncertainty"),
        ("21", "Developer guide", "Writing new Functors and parameters"),
    ]),
    ("Applied Topics (NB22-29)", [
        ("22", "Visualization", "Time series, scatter, conservation and graph plots"),
        ("23", "Multi-scale", "TemporalAggregator, TemporalDisaggregator, ScaleManager"),
        ("24", "Data I/O", "DataReader, DataWriter, DiagnosticWriter, IOHealthChecker"),
        ("25", "ML surrogates", "SurrogateFunctor, PyTorchSurrogate"),
        ("26", "Ensembles", "Weather generator, Euler-Maruyama SDE, Gillespie SSA, CRPS"),
        ("27", "Data assimilation", "3D-Var, 4D-Var, EnKF"),
        ("28", "Web and REST API", "create_web_app, create_api_app"),
        ("29", "Cross-domain", "ConservationGuardian, CrossDomainValidator, HealthChecker"),
    ]),
]

print()
print("=== What to Do Next ===")
print()
print("  Your RELIC installation is verified. Follow this learning path:")
for section, entries in LEARNING_PATH:
    print()
    print(f"  {section}:")
    for number, topic, content in entries:
        print(f"    {number} -> {topic:<18s} : {content}")



    RELIC Architecture Overview
    
    Core Framework (relic.core):
      - FunctorBase: Unified base class for all computational modules
      - RELICSimulator: Execution engine with dependency resolution
      - StateManager: Centralized state with conservation validation
      - Four Pillars: PhysicalField, BiologicalProcess, TransportMechanism, ScaleCoupling
    
    Scientific Modules (relic.scientific):
      - Atmosphere: Radiation, turbulence, chemistry, transport
      - Biosphere: Photosynthesis, respiration, growth, phenology, BVOC
      - Hydrosphere: ET, soil moisture, runoff, groundwater, snow
      - Pedosphere: Decomposition, nutrient cycling, carbon pools
      - Anthroposphere: Land use, management, emissions
      - Isotopes: Tracking and fractionation
      - Transport: Unified operators and boundary conditions
    
    Coupling (relic.coupling):
      - Internal: Cross-domain coupling (bio-atmo, soil-veg, water-carbon)
      - External: Third-party model adapter

## Summary

| Check | Status |
|---|---|
| Python & conda | See Step 1 |
| Dependencies | See Step 2 |
| RELIC package imports | See Step 3 |
| PyTorch & device | See Step 4 |
| Smoke test | See Step 5 |

If all checks show `[OK]`, your RELIC installation is ready.
Proceed to **NB01: Minimal RELIC Run** to begin the tutorial sequence.
